# Snowflake Intelligence Lab Guide

## What We Built So Far

In the **DT Lab Guide**, you built five silver Dynamic Iceberg Tables that aggregate bronze balloon game events into leaderboards, color stats, and real-time scores — all in open Iceberg format.

## What We'll Build Next

This notebook makes your silver data **AI-ready** by:

- Creating a **Semantic View** over the five silver tables (with Cortex Code or manually)
- Setting up an optional **Email tool** for the agent
- Configuring a **Snowflake Intelligence agent** that answers natural-language questions grounded in your data

**Prerequisites:**
- Silver Dynamic Iceberg Tables created and refreshed (from the DT Lab Guide)
- ACCOUNTADMIN role (for creating integrations and semantic views)
- A running warehouse

---

## Step 1: Configure Your Environment

Set the variables below to match your environment. All subsequent SQL cells reference these via Jinja templating.

> ### STOP — Update the variables below before proceeding!
>
> Make sure the values match your DT Lab Guide setup, then **run the cell below**.

In [ ]:
WAREHOUSE = 'DEFAULT_WH'
DB_NAME = 'summit26_ar103_balloon_silver'
SCHEMA_NAME = 'silver'
SEMANTIC_VIEW_NAME = 'balloon_game_semantic_view'
AGENT_NAME = 'balloon_game_agent'

---

## Step 2: Set Context

In [ ]:
%%sql -r use_role
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r use_wh
USE WAREHOUSE {{WAREHOUSE}};

In [ ]:
%%sql -r use_schema
USE DATABASE {{DB_NAME}};
USE SCHEMA {{SCHEMA_NAME}};

---

## Step 3: Create Semantic View

The SQL cell below creates a **Semantic View** over your five silver tables. It defines:
- **Tables** — business-friendly aliases, synonyms, primary keys, and unique constraints
- **Relationships** — how tables join (player → leaderboard, color → color_stats)
- **Facts** — raw numeric columns (score, pops, bonus hits) with natural-language synonyms
- **Dimensions** — categorical/temporal columns (player name, balloon color, time windows)
- **Metrics** — pre-defined aggregations (total score, avg points per pop, player count, etc.)
- **AI SQL Generation** — context hints that help Cortex Analyst generate accurate queries

The cell uses Jinja variables (`{{DB_NAME}}`, `{{SCHEMA_NAME}}`, `{{SEMANTIC_VIEW_NAME}}`) from **Step 1**.

> **Want to regenerate?** You can also use **Cortex Code** (Cmd+K / Ctrl+K) to regenerate or customize the Semantic View. Use this prompt:
>
> *I run a balloon popping game and I want anyone on my team to ask questions in plain English. My data lives in 5 silver tables — a leaderboard, color breakdown, live scores in 15-second windows, a detailed player+color+window view, and color performance trends. Create a semantic view with facts, dimensions, metrics, and AI SQL generation hints that helps an AI answer any question about players, scores, colors, or trends.*

Run the cell below to create the Semantic View.

In [ ]:
%%sql -r coco_semantic_view
-- TODO update with the SQL generated by cortex code

---

## Step 4: Verify the Semantic View

Confirm the view was created and inspect its structure.

In [ ]:
%%sql -r show_views
SHOW SEMANTIC VIEWS IN SCHEMA {{DB_NAME}}.{{SCHEMA_NAME}};

In [ ]:
%%sql -r desc_view
DESC SEMANTIC VIEW {{DB_NAME}}.{{SCHEMA_NAME}}.{{SEMANTIC_VIEW_NAME}};

---

## Step 5: (Optional) Set Up Email Tool

If you want the Intelligence agent to send query results via email, create the notification integration and stored procedure below. Skip this step if you only want interactive querying.

> **Requirement:** Your Snowflake user must have a verified email address for delivery to work.

In [ ]:
%%sql -r create_email_tool
-- Notification integration for email delivery
CREATE OR REPLACE NOTIFICATION INTEGRATION email_integration
  TYPE = EMAIL
  ENABLED = TRUE
  DEFAULT_SUBJECT = 'Balloon Game Analytics';

-- Stored procedure that the agent calls to send emails
CREATE OR REPLACE PROCEDURE {{DB_NAME}}.{{SCHEMA_NAME}}.send_email(
    recipient_email VARCHAR,
    subject VARCHAR,
    body VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    CALL SYSTEM$SEND_EMAIL(
        'email_integration',
        :recipient_email,
        :subject,
        :body,
        'text/html'
    );
    RETURN 'Email sent successfully to ' || :recipient_email;
END;

---

## Step 6: Create the Snowflake Intelligence Agent

Now configure an agent in the Snowsight UI that uses your Semantic View for natural-language querying.

**1. Navigate to the Agent admin page:**
- In Snowsight, go to **AI & ML → Agents**
- Confirm your role is set to **ACCOUNTADMIN** (top-right role selector)

**2. Create a new agent:**
- Click **+ Create agent**
- **Agent object name:** `balloon_game_agent`
- **Display name:** `Balloon Game Analytics`
- **Description:** *"Ask questions about balloon game player scores, color stats, and performance trends from the silver lakehouse tables."*
- Click **Create agent**

**3. Add the Cortex Analyst tool (Semantic View):**
- Select the **Tools** tab
- Find **Cortex Analyst** and click **+ Add**
- Choose **Semantic View** (not "Semantic model file")
- Select database: `{{DB_NAME}}`, schema: `{{SCHEMA_NAME}}`, view: `{{SEMANTIC_VIEW_NAME}}`
- For **Description**, click **Generate with Cortex** to auto-generate — or write: *"Queries structured balloon game data including player leaderboards, color stats, real-time scores, and performance trends. Use for any question about players, scores, colors, or time-based patterns."*
- Set the **Warehouse** to your lab warehouse

**4. (If you ran Step 5) Add the Email tool:**
- In the **Tools** tab, find **Custom Tools** and click **+ Add**
- Select database: `{{DB_NAME}}`, schema: `{{SCHEMA_NAME}}`, procedure: `send_email`
- Configure parameter descriptions (these guide the agent on how to use the tool):
  - **recipient_email:** *"If the email is not provided, send it to the current user's email address."*
  - **subject:** *"If subject is not provided, use 'Balloon Game Analytics'."*
  - **body:** *"If body is not provided, summarize the last question and use that as content for the email."*

**5. Add sample questions:**
- Select the **Voice** tab (or **Instructions** tab depending on your Snowsight version)
- Under **Sample questions**, add:
  - *"Who are the top 5 players by total score?"*
  - *"Which balloon color gives the best average points per pop?"*
  - *"Show me score trends over the last few time windows"*
  - *"How many total bonus pops have all players earned?"*
  - *"Email me a summary of the top 3 players"*

**6. Set orchestration instructions:**
- In the **Instructions** section, add the following orchestration instruction:
  - *"Whenever you can answer visually with a chart, always choose to generate a chart even if the user didn't specify to."*

**7. Save the agent** — Click **Save** in the top-right corner. The agent is now live.

---

## Step 7: Try It!

Open your agent and ask questions in natural language.

**Access the agent:**
- In Snowsight: **AI & ML → Snowflake Intelligence** → select `Balloon Game Analytics` from the agent picker
- Or go to [ai.snowflake.com](https://ai.snowflake.com) and select the agent

**Example questions to try:**

> Who are the top 5 players by total score?

> What's the most popular balloon color across all players?

> Which color gives the best average points per pop?

> Show me how player scores trend over time windows

> Which players have the most bonus pops as a percentage of total pops?

> Email me the top 3 players leaderboard

The agent routes your question to Cortex Analyst, which reads the Semantic View's metadata to generate accurate SQL against your silver Dynamic Iceberg Tables.

---

## What Just Happened?

You made your silver lakehouse data **AI-ready** using Snowflake Intelligence:

| Component | What it does |
|---|---|
| **Semantic View** | Maps business concepts (players, scores, colors) to physical tables and columns — the "brain" of the agent |
| **Cortex Analyst** | Converts natural-language questions into SQL grounded in the Semantic View |
| **Intelligence Agent** | Orchestrates tools (Analyst, Email) and provides a chat UI for business users |
| **Email Tool** | (Optional) Sends query results to users via email |

Key takeaways:
- **No code** — business users ask questions in plain English
- **Governed** — the agent respects RBAC; users only see data their role can access
- **Grounded** — answers come from SQL against real data, not hallucinated by an LLM
- **Same data** — the same Iceberg tables power dashboards (SiS), cross-engine queries (DuckDB), **and** AI analytics (Intelligence)

**Next up:** Build a Streamlit in Snowflake dashboard to visualize these silver tables — open the **SiS Dashboard** chapter in the quickstart guide.